In [ ]:
import matplotlib.pyplot as plt
import torch

from rlaopt.atoms import L1Norm
from rlaopt.expression import Variable
from rlaopt.linalg import NystromConfig
from rlaopt.solvers import ADMM, ADMMConfig, ADMMStoppingCriteria

In [ ]:
# torch.set_default_dtype(torch.float32)
torch.set_default_dtype(torch.float64)

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cuda:1"
# device = "cpu"

In [ ]:
# A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
# b = torch.tensor([5.0, 11.0])
# w = Variable((2, 2), name="w")
# x = Variable(torch.tensor([1.0, 2.0]), name="x")
# y = Variable(torch.tensor([3.0, 4.0]), name="y")
# z = Variable(torch.tensor([1.0, 2.0]), name="z")

# loss = 2 * (SumSquares(A @ x - b) + SumSquares(A @ y - b) +
#             SumSquares(x + y) + 2 * L1Norm(y, scaling=2.0) + L1Norm(x) + L1Norm(z))
# loss = loss.cuda()

In [ ]:
# m = 10000
# n = 100000
# reg = 1e-4

# X = torch.randn(m, n, device=device)  # data
# y = torch.randn(m, device=device)  # targets

# w = Variable(torch.zeros(n, device=device), name="w")  # parameters
# loss = SumSquares(X @ w - y) + reg * L1Norm(w)

In [ ]:
# n = 30000
# k = 300

# F = torch.randn(n, k, device=device)
# D = torch.diag(torch.rand(n, device=device)) * (k ** 0.5)
# D_sqrt = torch.sqrt(torch.diag(D))
# mu = torch.randn(n, device=device)
# gamma = 1.0

# x = Variable(torch.zeros(n, device=device), name="x")
# obj = -(mu * x).sum() + gamma/2 * (SumSquares(D_sqrt @ x) + SumSquares(F.T @ x))
# constraints = LinearEquality(x, torch.ones((1, n), device=device), torch.ones((1,), device=device)) + Box(x, lower=0.0)
# loss = obj + constraints
# loss.to(device)

In [ ]:
m = 10000
n = 100000

X = torch.randn(m, n, device=device)  # data
y = torch.randn(m, device=device)  # targets

w = Variable(torch.zeros(n, device=device), name="w")  # parameters
loss = L1Norm(X @ w - y)
loss = loss.to(device)

### Optimize objective

In [ ]:
solver = ADMM(
    obj=loss,
    config=ADMMConfig(
        rho=1e0, preconditioner_config=NystromConfig(rank_init=200, base_damping=0.0)
    ),
)
num_iterations = 300

In [ ]:
params = loss.variable_values
state = solver.init_state(params)
primal_residuals = []
dual_residuals = []
rhos = []
for i in range(num_iterations):
    params, state = solver.step(params, state)
    primal_residuals.append(state.primal_residual_norm)
    dual_residuals.append(state.dual_residual_norm)
    rhos.append(state.rho)
    if (i + 1) % 20 == 0:
        print(
            f"Iteration {i + 1}, primal residual: {state.primal_residual_norm}, dual residual: {state.dual_residual_norm}"
        )

In [ ]:
plt.plot(primal_residuals, label="Primal Residual")
plt.plot(dual_residuals, label="Dual Residual")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Residual Norm")
plt.title("ADMM Residuals over Iterations")
plt.legend()

In [ ]:
plt.plot(rhos, label="Rho")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Rho Value")
plt.title("ADMM Rho over Iterations")
plt.legend()

In [ ]:
final_result = solver.solve(stopping_criteria=ADMMStoppingCriteria(max_iters=500))

In [ ]:
final_result